### Imports

In [1]:
# importing libraries

import numpy as np
import pandas as pd
import joblib
import duckdb
import ast
from collections import Counter, defaultdict

# tensorflow
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.losses import BinaryCrossentropy

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
item_prop_df = pd.read_pickle('data/intermediate_files/item_properties.pkl')
session_df = pd.read_pickle('data/intermediate_files/sessionized_events.pkl')
cat_parent_map = pd.read_csv('data/intermediate_files/category_input.csv')

In [3]:
latest_cat_query = """
with 
dod_item_cat_map as
(
select 
    date(timestamp) as date, itemid, value,
    row_number() over(partition by date(timestamp), itemid order by timestamp desc) as rn
from 
    item_prop_df where property = 'categoryid'
qualify rn = 1
),

joined_df as (
select * from (
select
    a.*, coalesce(b.date, date(a.timestamp)) as categoryid_update_date, coalesce(b.value, a.category_id) as categoryid,
    row_number() over(partition by row_num order by b.date desc) as rn
from
(select *, row_number() over() as row_num from session_df) a
left join dod_item_cat_map b on a.itemid = b.itemid and b.date <= date(a.timestamp)
) t1 where rn = 1
)

select a.*, b.parentid as parent_category_id from joined_df a left join cat_parent_map b on a.categoryid = b.categoryid
"""

session_df = duckdb.sql(latest_cat_query).to_df()
session_df.drop(['rn', 'row_num', 'category_id'], axis=1, inplace=True)


### Helper functions

In [4]:
# convert x to list if not list
def as_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, set):
        return list(x)
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if x is None:
        return []
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            return [x]
    return [x]

In [5]:
# dedupes and keeps the order and converts input items to string
def dedupe_keep_order(items):
    seen, out = set(), []
    for x in as_list(items):
        try:
            if pd.isna(x):
                continue
        except Exception:
            pass
        x = str(int(x)) if isinstance(x, (int, np.integer, float, np.floating)) and not pd.isna(x) else str(x)
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

In [6]:
# converts float cols to string
def safe_id_string_col(s, unknown_value="unknown"):
    s_obj = s.astype("object")

    out = []
    for x in s_obj:
        if pd.isna(x):
            out.append(unknown_value)
        else:
            try:
                xf = float(x)
                if xf.is_integer():
                    out.append(str(int(xf)))
                else:
                    out.append(str(x))
            except Exception:
                out.append(str(x))

    return pd.Series(out, index=s.index).replace({
        "nan": unknown_value,
        "None": unknown_value,
        "<NA>": unknown_value
    })

### Train and val/test data split

In [7]:
artifact_ranges = {
    1: ("2015-05-03", "2015-06-01"),
    2: ("2015-05-15", "2015-06-15"),
    3: ("2015-06-01", "2015-07-01"),
    4: ("2015-06-15", "2015-07-15"),
    5: ("2015-07-01", "2015-08-01"),
    6: ("2015-07-15", "2015-08-15"),
    7: ("2015-08-01", "2015-09-01"),
}

query_ranges = {
    1: ("2015-06-01", "2015-06-15"),
    2: ("2015-06-15", "2015-07-01"),
    3: ("2015-07-01", "2015-07-15"),
    4: ("2015-07-15", "2015-08-01"),
    5: ("2015-08-01", "2015-08-15"),
    6: ("2015-08-15", "2015-09-01"),
    7: ("2015-09-01", "2015-09-15"),
}


# for v1 we will keep everything and hence comment down the below optimization - to have atleast 2L training rows
# later we might, we might keep only sessions where orders/atc happened - we can change this if needed and then sample the train data differently
# engaged_sns = session_df.loc[
#     session_df["event"].isin(["addtocart", "transaction"]),
#     ["visitorid", "session_id"]
# ].drop_duplicates()

# train_reduced_sess_df = session_df.merge(
#     engaged_sns,
#     on=["visitorid", "session_id"],
#     how="inner"
# )

arft_time, block_time = {}, {}
block_num = 7

for i in range(1, block_num+1):
    a_start, a_end = artifact_ranges[i]
    q_start, q_end = query_ranges[i]

    arft_time[i] = (
        (session_df["timestamp"] >= a_start) &
        (session_df["timestamp"] < a_end)
    )

    query_source_df = session_df #train_reduced_sess_df if i <= 5 else session_df

    block_time[i] = (
        (query_source_df["timestamp"] >= q_start) &
        (query_source_df["timestamp"] < q_end)
    )

In [8]:
# query table for past data

def build_query_table_past(query_df, session_df, k=25, C=5):

    output_rows = []

    # group once for speed
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        udf = user_groups[row["visitorid"]]

        hist_df = udf[udf.timestamp < row['timestamp']]
        hist_df['hour'] = hist_df["timestamp"].dt.hour.astype(str)
        hist_df["day"] = hist_df["timestamp"].dt.day_name().astype(str)

        last_k_items = set(hist_df['itemid'].drop_duplicates(keep='last').tail(k).tolist())
        last_C_categories = set(hist_df['categoryid'].drop_duplicates(keep='last').tail(C).tolist())

        top_10_interacted_items = hist_df.groupby('itemid').size().sort_values(ascending=False).head(10).index.tolist()
        top_5_interacted_cats = hist_df.groupby('categoryid').size().sort_values(ascending=False).head(5).index.tolist()

        favourite_3_hours = hist_df.groupby('hour').size().sort_values(ascending=False).head(3).index.tolist()
        favourite_day = hist_df.groupby('day').size().sort_values(ascending=False).index[0]
        
        event_grouped = hist_df.groupby("event")

        past_events = event_grouped.size()

        # purchased items
        pre_purchased = set(event_grouped['itemid'].apply(list).get('transaction',[]))

        output_rows.append({
            "query_id": query_id,
            "visitorid": row["visitorid"],
            "session_id": row["session_id"],
            "sequence_no": row["sequence_no"],
            "event_pos_in_session": row["event_pos_in_session"],
            "query_time": row["timestamp"],
            "current_event": row["event"],
            "past_view_count": past_events.get("view", 0),
            "past_atc_count": past_events.get("addtocart", 0),
            "past_order_count": past_events.get("transaction", 0),
            "previous_sessions": row["session_id"] - 1,
            "last_k_items": last_k_items,
            "last_C_categories": last_C_categories,
            "pre_purchased": pre_purchased,
            "top_10_interacted_items": top_10_interacted_items,
            "top_5_interacted_cats": top_5_interacted_cats,
            "favourite_3_hours": favourite_3_hours,
            "favourite_day": favourite_day
        })

    return pd.DataFrame(output_rows)

In [9]:
def build_query_table_fut(query_df, session_fut_df, query_type):

    output_rows = []

    # group once for speed - past data
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_fut_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        user_df = user_groups[row["visitorid"]]

        curr_sess = row["session_id"]
        curr_ts = row["timestamp"]

        # future positives within same session, using only all future interactions in the same session
        fut_df = user_df[
            (user_df["session_id"] == curr_sess) &
            (user_df["timestamp"] >= curr_ts)
            # & (user_df["event"] == "view")
        ]

        view_items = list(dict.fromkeys(fut_df.loc[fut_df.event == "view", "itemid"].tolist()))
        atc_items = list(dict.fromkeys(fut_df.loc[fut_df.event == "addtocart", "itemid"].tolist()))
        order_items = list(dict.fromkeys(fut_df.loc[fut_df.event == "transaction", "itemid"].tolist()))
        interaction_items = list(dict.fromkeys(fut_df["itemid"].tolist()))
        engagement_items = list(dict.fromkeys(atc_items + order_items))

        # while training, only keep those queries where we have orders in the future positives
        # if query_type == 'train' and ((len(interaction_items) == 0) or (len(engagement_items) == 0)):
        #     continue

        # if query_type != 'train' and (len(interaction_items) == 0):
        #     continue

        if len(interaction_items) == 0:
            continue

        output_rows.append({
            "query_id": query_id,
            "future_positives": interaction_items,
            "future_view_items": view_items,
            "future_atc_items": atc_items,
            "future_order_items": order_items,
            "future_engagement_items": engagement_items
        })

    return pd.DataFrame(output_rows)

In [10]:
# query_inputs = {}
# sample_rows = 30000
# block_num = 7

# for i in range(1, block_num+1):
#     query_source_df = session_df #train_reduced_sess_df if i <= 5 else session_df
#     df_query = query_source_df.loc[block_time[i]].copy()

#     not_first_new_session = ((df_query["is_new_session"] == 1) & (df_query["session_id"] > 1))

#     valid_q = df_query.loc[not_first_new_session].drop_duplicates(subset=["visitorid", "session_id", "timestamp"]).copy()

#     # sampling only val and test for now - train will be sampled later basis positives
#     if i > 5:
#         query_base = valid_q.sample(n=min(sample_rows, valid_q.shape[0]),random_state=42 + i)
#     else:
#         query_base = valid_q

#     print('query base shape: ', query_base.shape)
    
#     query_ip_past = build_query_table_past(query_base, session_df)

#     query_type = "train" if i <= 5 else "test"
#     query_ip_fut = build_query_table_fut(query_base, df_query, query_type)

#     query_input = pd.merge(left=query_ip_past, right=query_ip_fut, on="query_id", how="inner")

#     query_inputs[i] = query_input.copy()
#     query_inputs[i]['block_id'] = i

#     print('query input shape: ', query_inputs[i].shape)

In [11]:
# saving the intermediate query inputs so that i dont run it again until absolutely needed - just in case

# for i in range(1, block_num+1):
#     query_inputs[i]['block_id'] = i

# pd.concat(query_inputs.values(), axis=0, ignore_index=True).to_pickle("data/intermediate_files/deep_cg_query_ips.pkl")

# retrieving the query inputs from the saved csv
all_query_df = pd.read_pickle("data/intermediate_files/deep_cg_query_ips.pkl")

query_inputs = {}

for i in all_query_df.block_id.unique():
    query_inputs[i] = all_query_df[all_query_df.block_id == i]

In [12]:
# adding other item - related columns
def make_two_tower_query_df(query_outputs, split_name="train"):

    all_dfs = []
    offset = 0

    for df in query_outputs:
        qdf = df.copy().reset_index(drop=True)

        qdf["split"] = split_name
        qdf["local_query_id"] = qdf["query_id"] if "query_id" in qdf.columns else qdf.index
        qdf["tt_query_id"] = np.arange(offset, offset + len(qdf))
        offset += len(qdf)

        if "query_time" not in qdf.columns:
            qdf["query_time"] = qdf["timestamp"]

        # Time features
        qdf["query_time"] = pd.to_datetime(qdf["query_time"])
        qdf["day"] = qdf["query_time"].dt.day_name().astype(str)
        qdf["hour_of_day"] = qdf["query_time"].dt.hour.astype(str)

        # Log features for skewed numeric columns
        for col in ["past_view_count", "past_atc_count", "past_order_count", "previous_sessions"]:
            qdf[col] = qdf[col].fillna(0)
            qdf[f"log_{col}"] = np.log1p(qdf[col].astype(float))

        # Create simple user-history bucket from past view count.
        if "past_view_count" in qdf.columns:
            qdf["user_history_bucket"] = pd.cut(
                qdf["past_view_count"],
                bins=[-1, 0, 5, 20, 100, np.inf],
                labels=["no_history", "very_low", "low", "medium", "warm"]
            ).astype(str)

        # Basic null handling
        qdf["user_history_bucket"] = qdf["user_history_bucket"].fillna("unknown").astype(str)
        qdf["user_history_bucket"] = safe_id_string_col(qdf["user_history_bucket"])

        list_cols = [
            "last_k_items", "last_C_categories", 'pre_purchased', 
            'favourite_3_hours', 'top_10_interacted_items', 'top_5_interacted_cats',
            "future_positives", "future_view_items",
            "future_atc_items",
            "future_order_items", "future_engagement_items"
        ]

        for col in list_cols:
            if col in qdf.columns:
                qdf[col] = qdf[col].apply(dedupe_keep_order)
            else:
                qdf[col] = [[] for _ in range(len(qdf))]

        keep_cols = [
            "tt_query_id", "local_query_id", "block_id", "split",
            "visitorid", "session_id", "query_time",
            "last_k_items", "last_C_categories",
            "log_past_view_count", "log_past_atc_count", "log_past_order_count", "log_previous_sessions",
            'pre_purchased', 'favourite_3_hours', 'top_10_interacted_items', 'top_5_interacted_cats',
            "user_history_bucket", "day", "hour_of_day", 'favourite_day',
            "future_positives", "future_view_items", 
            "future_atc_items", "future_order_items", "future_engagement_items"
        ]

        keep_cols = [c for c in keep_cols if c in qdf.columns]
        all_dfs.append(qdf[keep_cols])

    return pd.concat(all_dfs, ignore_index=True)

In [13]:
train_query_dfs = {}

for i in [1, 2, 3, 4, 5]:
    train_query_dfs[i] = make_two_tower_query_df(
    [query_inputs[i]],
    split_name="train"
    )

val_query_df = make_two_tower_query_df(
    [query_inputs[6]],
    split_name="val"
)

test_query_df = make_two_tower_query_df(
    [query_inputs[7]],
    split_name="test"
)

### Two Tower

#### Active Item Universe

In [14]:
def get_item_cat_map(df):

    tp_df = df.copy()

    return duckdb.sql("""
    select itemid, categoryid, parent_category_id
    from
    (select *, row_number() over(partition by itemid order by timestamp desc) as rn
    from
    (select distinct timestamp, itemid, categoryid, parent_category_id from tp_df) t1
    ) t2
    where rn = 1
    group by 1,2,3
    """).to_df()


In [15]:
def get_cum_bucket(df, col_name):

    # create the orders bucket
    tp_df = df.copy()

    bucket_query = f"""
    select *,
    case when cum_oc <= 0.1 then '0to10'
        when cum_oc <= 0.2 then '10to20'
        when cum_oc <= 0.3 then '20to30'
        when cum_oc <= 0.4 then '30to40'
        when cum_oc <= 0.5 then '40to50'
        when cum_oc <= 0.6 then '50to60'
        when cum_oc <= 0.7 then '60to70'
        when cum_oc <= 0.8 then '70to80'
        when cum_oc <= 0.9 then '80to90'
    else '90to100' end as cum_bucket
    from (
    select
        *, sum({col_name}) over(order by {col_name} desc rows between unbounded preceding and current row)*1.0000/sum({col_name}) over() as cum_oc 
    from
        tp_df
        ) as t1
    """

    return duckdb.sql(bucket_query).to_df()


In [16]:
## platform popularity based on orders basis 7 and 30 day popularity

def popularity_cg_base(size, session_past_df, d):

    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    orders_past_df = session_past_df.loc[((session_past_df.transactionid.notnull()) & (session_past_df.timestamp >= min_time))]

    item_pop = orders_past_df.groupby('itemid').size().sort_values(ascending=False).reset_index()
    item_pop.columns = ['itemid', 'orders']

    return item_pop.itemid.tolist()[:size], item_pop

In [17]:
## scores item basis their spread of users, interactions

def generality_CG(session_past_df, d, size):
    
    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    hist_df = session_past_df[(session_past_df.timestamp >= min_time)]

    generality_query = """
    select
        itemid

        , count(distinct visitorid) as unique_users
        , count(distinct cast(visitorid as varchar) || '-' || cast(session_id as varchar)) as unique_sessions
        , count(distinct date(timestamp)) as unique_days

        , (
            0.1*sum(case when event='view' then 1 else 0 end) 
            + 0.3*sum(case when event='addtocart' then 1 else 0 end) 
            + 0.6*sum(case when event='transaction' then 1 else 0 end)
        ) as weighted_interaction_score

    from hist_df
    group by 1
    """

    item_stats = duckdb.sql(generality_query).to_df()

    for col in ['unique_users', 'unique_sessions', 'unique_days', 'weighted_interaction_score']:

        item_stats[f'{col}_pct'] = item_stats[col].rank(pct=True)

    item_stats['generality_score'] = (
        0.5*item_stats['weighted_interaction_score_pct']
        + 0.25*item_stats['unique_users_pct']
        + 0.15*item_stats['unique_sessions_pct']
        + 0.10*item_stats['unique_days_pct']
    )

    item_stats = duckdb.sql('select *, row_number() over(order by generality_score desc) as item_rank from item_stats').to_df()

    return item_stats.sort_values('generality_score', ascending=False)['itemid'].to_list()[:size], item_stats



In [18]:
def get_active_items(df, max_items=50000):

    df = df.copy()

    item_stats = df.groupby("itemid").agg(
        view_count=("event", lambda x: (x == "view").sum()),
        atc_count=("event", lambda x: (x == "addtocart").sum()),
        order_count=("event", lambda x: (x == "transaction").sum()),
        total_interactions=("event", "size")
    ).reset_index()

    item_stats["weighted_score"] = (
        0.1 * item_stats["view_count"]
        + 0.3 * item_stats["atc_count"]
        + 0.6 * item_stats["order_count"]
    )

    active_items = (
        item_stats
        .sort_values("weighted_score", ascending=False)
        .head(max_items)
        .copy()
    )

    active_items["neg_sampling_weight"] = active_items["weighted_score"].clip(lower=1)
    active_items["neg_sampling_weight"] = active_items["neg_sampling_weight"] / active_items["neg_sampling_weight"].sum()

    # item cat mapping
    item_cap_map_df = get_item_cat_map(df)

    # get item popularity buckets
    _, item_pop_df = popularity_cg_base(50, session_past_df=df, d=30)
    item_pop_df = get_cum_bucket(item_pop_df, 'orders')

    # get item generality buckets
    _, item_gen_df = generality_CG(session_past_df=df, d=30, size=100)
    item_gen_df = get_cum_bucket(item_gen_df, 'generality_score')

    # join to get all columns
    active_items = active_items.merge(item_cap_map_df, on="itemid", how="left")

    active_items = active_items.merge(item_pop_df, on="itemid", how="left")
    active_items['cum_bucket'] = active_items['cum_bucket'].fillna('90to100')
    active_items.rename(columns={'cum_bucket':'popularity_bucket'}, inplace=True)

    active_items = active_items.merge(item_gen_df, on="itemid", how="left")
    active_items['cum_bucket'] = active_items['cum_bucket'].fillna('90to100')
    active_items.rename(columns={'cum_bucket':'generality_bucket'}, inplace=True)

    cat_cols = ["categoryid", "parent_category_id", "popularity_bucket", "generality_bucket"]

    # null handling
    for col in cat_cols:
        active_items[col] = safe_id_string_col(active_items[col])

    active_items["itemid"] = active_items["itemid"].astype(str)

    return active_items


#### Two tower input data

In [34]:
# We keep all ATC/order positives and sample some view-only positives.
def create_two_tower_pairs(BW_query_df, neg_per_positive=20, view_only_keep_rate=0.5, random_state=42):
    rng = np.random.default_rng(random_state)

    # find out the active items block wise
    block_id = BW_query_df.block_id.values[0] # since block id remains same for the df
    active_item_df = get_active_items(session_df[arft_time[block_id]])

    item_pool = active_item_df["itemid"].astype(str).values
    item_probs = active_item_df["neg_sampling_weight"].values
    item_meta_lookup = active_item_df.set_index("itemid").to_dict("index")
    active_set = set(item_pool)


    rows = []

    for _, q in BW_query_df.iterrows():
        view_set = set(dedupe_keep_order(q["future_view_items"]))
        atc_set = set(dedupe_keep_order(q["future_atc_items"]))
        order_set = set(dedupe_keep_order(q["future_order_items"]))

        positive_items = list((view_set | atc_set | order_set) & active_set)

        if len(positive_items) == 0:
            continue

        kept_positive_items = []

        for itemid in positive_items:
            y_view = int(itemid in view_set)
            y_atc = int(itemid in atc_set)
            y_order = int(itemid in order_set)

            is_view_only = (y_view == 1 and y_atc == 0 and y_order == 0)

            if is_view_only and rng.random() > view_only_keep_rate:
                continue

            kept_positive_items.append(itemid)

            # print(item_meta_lookup)
            meta = item_meta_lookup[itemid]

            rows.append({
                "tt_query_id": q["tt_query_id"],
                "block_id": block_id,
                "itemid": itemid,
                "y_view": y_view,
                "y_atc": y_atc,
                "y_order": y_order,
                "label_any": 1,

                'pre_purchased': q["pre_purchased"], 
                "last_k_items": q["last_k_items"],
                "last_C_categories": q["last_C_categories"],
                "log_past_view_count": q["log_past_view_count"],
                "log_past_atc_count": q["log_past_atc_count"],
                "log_past_order_count": q["log_past_order_count"],
                'log_previous_sessions': q["log_previous_sessions"],
                'top_10_interacted_items': q["top_10_interacted_items"], 
                'top_5_interacted_cats': q["top_5_interacted_cats"],

                'favourite_3_hours': q["favourite_3_hours"],
                "user_history_bucket": q["user_history_bucket"],
                "day": q["day"],
                "hour_of_day": q["hour_of_day"],
                'favourite_day': q["favourite_day"],

                # item features
                "categoryid": meta.get("categoryid", "unknown"),
                "parent_category_id": meta.get("parent_category_id", "unknown"),
                "popularity_bucket": meta.get("popularity_bucket", "unknown"),
                "generality_bucket": meta.get("generality_bucket", "unknown")
            })

        if len(kept_positive_items) == 0:
            continue

        true_all = set(positive_items)
        n_neg = len(kept_positive_items) * neg_per_positive

        valid_mask = np.array([item not in true_all for item in item_pool])
        neg_pool = item_pool[valid_mask]
        neg_probs = item_probs[valid_mask]
        neg_probs = neg_probs / neg_probs.sum()

        replace = len(neg_pool) < n_neg
        neg_items = rng.choice(neg_pool, size=n_neg, replace=replace, p=neg_probs)

        for itemid in neg_items:
            meta = item_meta_lookup[itemid]

            rows.append({
                "tt_query_id": q["tt_query_id"],
                "block_id": block_id,
                "itemid": itemid,
                "y_view": 0,
                "y_atc": 0,
                "y_order": 0,
                "label_any": 0,

                'pre_purchased': q["pre_purchased"], 
                "last_k_items": q["last_k_items"],
                "last_C_categories": q["last_C_categories"],
                "log_past_view_count": q["log_past_view_count"],
                "log_past_atc_count": q["log_past_atc_count"],
                "log_past_order_count": q["log_past_order_count"],
                'log_previous_sessions': q["log_previous_sessions"],
                'top_10_interacted_items': q["top_10_interacted_items"], 
                'top_5_interacted_cats': q["top_5_interacted_cats"],

                'favourite_3_hours': q["favourite_3_hours"],
                "user_history_bucket": q["user_history_bucket"],
                "day": q["day"],
                "hour_of_day": q["hour_of_day"],
                'favourite_day': q["favourite_day"], 

                # item features
                "categoryid": meta.get("categoryid", "unknown"),
                "parent_category_id": meta.get("parent_category_id", "unknown"),
                "popularity_bucket": meta.get("popularity_bucket", "unknown"),
                "generality_bucket": meta.get("generality_bucket", "unknown")
            })

    pair_df = pd.DataFrame(rows)

    for col in ["itemid", "categoryid", "parent_category_id", "popularity_bucket", "generality_bucket",
                "user_history_bucket", "day", "hour_of_day", 'favourite_day']:
        pair_df[col] = safe_id_string_col(pair_df[col])

    return pair_df


In [36]:
# pair dfs for train/val/test
offset = 0

tt_train_pair_dfs = []
for i in train_query_dfs.keys():

    train_query_dfs[i]["tt_query_id"] = np.arange(offset, offset + len(train_query_dfs[i]))
    offset += len(train_query_dfs[i])
    
    tt_train_pair_dfs.append(create_two_tower_pairs(train_query_dfs[i], neg_per_positive=20, view_only_keep_rate=0.5, random_state=42+i))

tt_train_pair_df = pd.concat(tt_train_pair_dfs, axis=0, ignore_index=True)

In [37]:
tt_train_pair_df.columns

Index(['tt_query_id', 'block_id', 'itemid', 'y_view', 'y_atc', 'y_order',
       'label_any', 'pre_purchased', 'last_k_items', 'last_C_categories',
       'log_past_view_count', 'log_past_atc_count', 'log_past_order_count',
       'log_previous_sessions', 'top_10_interacted_items',
       'top_5_interacted_cats', 'favourite_3_hours', 'user_history_bucket',
       'day', 'hour_of_day', 'favourite_day', 'categoryid',
       'parent_category_id', 'popularity_bucket', 'generality_bucket'],
      dtype='str')

In [38]:
tt_train_pair_df.head()

,tt_query_id,block_id,itemid,y_view,y_atc,y_order,label_any,pre_purchased,last_k_items,last_C_categories,...,top_5_interacted_cats,favourite_3_hours,user_history_bucket,day,hour_of_day,favourite_day,categoryid,parent_category_id,popularity_bucket,generality_bucket
0,1,1,425186,1,0,0,1,[],[425186],[1143],...,[1143],"[17, 22, 21]",low,Monday,1,Sunday,1143,1424,90to100,0to10
1,1,1,320494,0,0,0,0,[],[425186],[1143],...,[1143],"[17, 22, 21]",low,Monday,1,Sunday,819,798,20to30,0to10
2,1,1,257929,0,0,0,0,[],[425186],[1143],...,[1143],"[17, 22, 21]",low,Monday,1,Sunday,1221,1426,90to100,70to80
3,1,1,348875,0,0,0,0,[],[425186],[1143],...,[1143],"[17, 22, 21]",low,Monday,1,Sunday,1253,127,90to100,20to30
4,1,1,134255,0,0,0,0,[],[425186],[1143],...,[1143],"[17, 22, 21]",low,Monday,1,Sunday,1051,955,90to100,0to10


#### Active item Coverage Check

In [22]:
def active_universe_coverage(query_df, active_item_df, true_col):
    active_set = set(active_item_df["itemid"].astype(str))

    n_queries = 0
    hit_queries = 0
    total_items = 0
    covered_items = 0

    for _, row in query_df.iterrows():
        trues = dedupe_keep_order(row[true_col])
        if len(trues) == 0:
            continue

        n_queries += 1
        total_items += len(trues)

        covered = [x for x in trues if x in active_set]
        covered_items += len(covered)

        if len(covered) > 0:
            hit_queries += 1

    return {
        "true_col": true_col,
        "query_coverage": hit_queries / n_queries if n_queries else 0,
        "item_coverage": covered_items / total_items if total_items else 0,
        "n_queries": n_queries,
        "n_items": total_items
    }

In [23]:
for i in [1, 2, 3, 4, 5, 6, 7]:
    active_i = get_active_items(session_df[arft_time[i]])

    qdf = (
        train_query_dfs[i]
        if i <= 5
        else val_query_df if i == 6
        else test_query_df
    )

    print("Block", i)
    print(active_universe_coverage(qdf, active_i, "future_positives"))
    print(active_universe_coverage(qdf, active_i, "future_engagement_items"))

Block 1
{'true_col': 'future_positives', 'query_coverage': 0.792021870774296, 'item_coverage': 0.7638587770785963, 'n_queries': 34018, 'n_items': 56733}
{'true_col': 'future_engagement_items', 'query_coverage': 0.8890356671070013, 'item_coverage': 0.834936514043863, 'n_queries': 1514, 'n_items': 2599}
Block 2
{'true_col': 'future_positives', 'query_coverage': 0.7721032143772872, 'item_coverage': 0.7402717140765247, 'n_queries': 42901, 'n_items': 73754}
{'true_col': 'future_engagement_items', 'query_coverage': 0.8756319514661274, 'item_coverage': 0.8001122964626615, 'n_queries': 1978, 'n_items': 3562}
Block 3
{'true_col': 'future_positives', 'query_coverage': 0.7695561331091066, 'item_coverage': 0.7284637177161872, 'n_queries': 39246, 'n_items': 66713}
{'true_col': 'future_engagement_items', 'query_coverage': 0.8782760629004077, 'item_coverage': 0.8160810383032605, 'n_queries': 1717, 'n_items': 3159}
Block 4
{'true_col': 'future_positives', 'query_coverage': 0.7503391341420875, 'item_co

### Build Model

In [39]:
active_items_by_block = {}

for i in range(1, 8):
    active_items_by_block[i] = get_active_items(session_df[arft_time[i]]).copy()
    active_items_by_block[i]["itemid"] = safe_id_string_col(active_items_by_block[i]["itemid"])

    for col in ["categoryid", "parent_category_id", "popularity_bucket", "generality_bucket"]:
        active_items_by_block[i][col] = safe_id_string_col(active_items_by_block[i][col])

    print(i, active_items_by_block[i].shape)


1 (50000, 24)
2 (50000, 24)
3 (50000, 24)
4 (50000, 24)
5 (50000, 24)
6 (50000, 24)
7 (50000, 24)


#### Fix length arrays

In [41]:
# helper

def pad_or_truncate(items, max_len, pad_token="__PAD__"):
    items = dedupe_keep_order(items)
    items = items[-max_len:]
    if len(items) < max_len:
        items = items + [pad_token] * (max_len - len(items))
    return items


In [42]:
# helper - explodes list values in a columns - row by row and return an exhaustive list

def explode_list_values(series):
    vals = []
    for x in series:
        vals.extend(dedupe_keep_order(x))
    return vals


In [43]:
# helper

def get_active_values(active_items_by_block, block_ids, col):
    vals = []

    for b in block_ids:
        if col in active_items_by_block[b].columns:
            vals.extend(active_items_by_block[b][col].astype(str).tolist())

    return vals


In [44]:
# fixed length arrays - for embeddings of array features

item_len_dic = {'last_k_items':20, 'last_C_categories':5, 'pre_purchased':20, 
                'favourite_3_hours':3, 'top_10_interacted_items':10, 'top_5_interacted_cats':5}

PAD_TOKEN = "__PAD__"
array_cols = list(item_len_dic.keys())

def add_fixed_history_arrays(df, array_cols):
    df = df.copy()

    for col in array_cols:
        df[col] = df[col].apply(
            lambda x: pad_or_truncate(x, item_len_dic[col], PAD_TOKEN)
        )

    return df


#### building vocabularies & lookup layer

In [45]:
def build_vocabularies(train_pair_df, active_items_by_block, train_blocks=[1, 2, 3, 4, 5]):
    vocab = {}

    active_itemids = get_active_values(active_items_by_block, train_blocks, "itemid")
    active_categories = get_active_values(active_items_by_block, train_blocks, "categoryid")
    active_parent_categories = get_active_values(active_items_by_block, train_blocks, "parent_category_id")
    active_pop_buckets = get_active_values(active_items_by_block, train_blocks, "popularity_bucket")
    active_gen_buckets = get_active_values(active_items_by_block, train_blocks, "generality_bucket")

    # Item-list query features
    item_array_cols = ["last_k_items", "pre_purchased", "top_10_interacted_items"]

    for col in item_array_cols:
        vals = [x for x in explode_list_values(train_pair_df[col]) if x != PAD_TOKEN]
        vocab[col] = sorted(set(vals) | set(active_itemids))

    # Category-list query features
    cat_array_cols = ["last_C_categories", "top_5_interacted_cats"]

    for col in cat_array_cols:
        vals = [x for x in explode_list_values(train_pair_df[col]) if x != PAD_TOKEN]
        vocab[col] = sorted(set(vals) | set(active_categories))

    # Favourite hours list
    fav_hours = [x for x in explode_list_values(train_pair_df["favourite_3_hours"]) if x != PAD_TOKEN]
    vocab["favourite_3_hours"] = sorted(set(map(str, fav_hours)) | set(map(str, range(24))))

    # Item tower vocab
    vocab["itemid"] = sorted(set(active_itemids))
    vocab["categoryid"] = sorted(set(active_categories))
    vocab["parent_category_id"] = sorted(set(active_parent_categories))
    vocab["popularity_bucket"] = sorted(set(active_pop_buckets))
    vocab["generality_bucket"] = sorted(set(active_gen_buckets))

    # Scalar query-context vocab
    vocab["user_history_bucket"] = sorted(set(train_pair_df["user_history_bucket"].astype(str)))
    vocab["day"] = sorted(set(train_pair_df["day"].astype(str)))
    vocab["hour_of_day"] = sorted(set(train_pair_df["hour_of_day"].astype(str)) | set(map(str, range(24))))
    vocab["favourite_day"] = sorted(set(train_pair_df["favourite_day"].astype(str)))

    return vocab


In [46]:
unmasked_cat_cols = [
        "itemid", "categoryid", "parent_category_id",
        "popularity_bucket", "generality_bucket",'favourite_day',
        "user_history_bucket", "day", "hour_of_day"]

masked_array_cols = ["last_k_items", "last_C_categories", "pre_purchased",
    "favourite_3_hours", "top_10_interacted_items", "top_5_interacted_cats",]

def build_lookup_layers(vocab, masked_array_cols, unmasked_cat_cols):
    lookups = {}

    for col in masked_array_cols:
        lookups[col] = layers.StringLookup(
            vocabulary=vocab[col],
            mask_token=PAD_TOKEN,
            num_oov_indices=1,
            output_mode="int",
            name=f"{col}_lookup"
        )

    for col in unmasked_cat_cols:
        lookups[col] = layers.StringLookup(
            vocabulary=vocab[col],
            mask_token=None,
            num_oov_indices=1,
            output_mode="int",
            name=f"{col}_lookup"
        )

    return lookups


In [47]:
tt_train_pair_df = add_fixed_history_arrays(tt_train_pair_df, array_cols)

vocab = build_vocabularies(
    train_pair_df=tt_train_pair_df,
    active_items_by_block=active_items_by_block,
    train_blocks=[1, 2, 3, 4, 5]
)

for k, v in vocab.items():
    print(k, len(v))

last_k_items 101140
pre_purchased 88118
top_10_interacted_items 97685
last_C_categories 1059
top_5_interacted_cats 1045
favourite_3_hours 24
itemid 88065
categoryid 1026
parent_category_id 272
popularity_bucket 10
generality_bucket 10
user_history_bucket 5
day 7
hour_of_day 24
favourite_day 7


In [48]:
def pair_df_checks(df, name):
    print(f"\n{name}")
    print("shape:", df.shape)
    print("queries:", df["tt_query_id"].nunique())
    print("positive rows:", df["label_any"].sum())
    print("positive rate:", df["label_any"].mean())
    print("view positive rate:", df["y_view"].mean())
    print("atc positive rate:", df["y_atc"].mean())
    print("order positive rate:", df["y_order"].mean())
    print("rows/query:")
    print(df.groupby("tt_query_id").size().describe())

    if "block_id" in df.columns:
        print("block distribution:")
        print(df["block_id"].value_counts().sort_index())

pair_df_checks(tt_train_pair_df, "TRAIN PAIRS")


TRAIN PAIRS
shape: (2749551, 25)
queries: 91419
positive rows: 130931
positive rate: 0.047619047619047616
view positive rate: 0.04647049645560312
atc positive rate: 0.004248693695807061
order positive rate: 0.0022214536118806308
rows/query:
count    91419.000000
mean        30.076363
std         42.147368
min         21.000000
25%         21.000000
50%         21.000000
75%         21.000000
max       1869.000000
dtype: float64
block distribution:
block_id
1    477183
2    599760
3    540435
4    673029
5    459144
Name: count, dtype: int64


In [49]:
lookup_layers = build_lookup_layers(
    vocab=vocab,
    masked_array_cols=masked_array_cols,
    unmasked_cat_cols=unmasked_cat_cols
)

#### Tensor flow Data set creation

In [50]:
query_numeric_cols = ["log_past_view_count", "log_past_atc_count", "log_past_order_count", "log_previous_sessions",]

query_scalar_cat_cols = [ "user_history_bucket", "day", "hour_of_day", "favourite_day",]

item_cat_cols = ["itemid", "categoryid", "parent_category_id", "popularity_bucket", "generality_bucket",]

def clean_pair_df_for_dataset(df):
    df = df.copy()

    for col in array_cols:
        df[col] = df[col].apply(lambda x: pad_or_truncate(x, item_len_dic[col], PAD_TOKEN))

    for col in query_numeric_cols:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = df[col].fillna(0).astype("float32")

    for col in query_scalar_cat_cols + item_cat_cols:
        df[col] = safe_id_string_col(df[col])

    return df


In [51]:
def make_two_tower_dataset(df, batch_size=512, shuffle=False, random_state=42):
    df = clean_pair_df_for_dataset(df)

    feature_dict = {}

    # np.stack combines list to make list of list
    for col in array_cols:
        feature_dict[col] = np.stack(df[col].values).astype(str)

    feature_dict["query_numeric"] = df[query_numeric_cols].astype("float32").values

    for col in query_scalar_cat_cols:
        feature_dict[col] = df[col].astype(str).values

    for col in item_cat_cols:
        feature_dict[col] = df[col].astype(str).values

    labels = df[["y_view", "y_atc", "y_order"]].astype("float32").values

    ds = tf.data.Dataset.from_tensor_slices((feature_dict, labels))

    if shuffle:
        ds = ds.shuffle(
            buffer_size=min(len(df), 100_000),
            seed=random_state
        )

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
train_ds = make_two_tower_dataset(
    tt_train_pair_df,
    batch_size=512,
    shuffle=True,
    random_state=42
)

# If you have val pairs and want pair-level validation loss:
tt_val_pair_df = create_two_tower_pairs(val_query_df, neg_per_positive=20, view_only_keep_rate=0.5, random_state=42)
val_ds = make_two_tower_dataset(
    tt_val_pair_df,
    batch_size=512,
    shuffle=False
)

#### Average pooling

In [53]:
class MaskedAveragePooling(layers.Layer):
    def call(self, inputs):
        emb, ids = inputs

        mask = tf.cast(tf.not_equal(ids, 0), tf.float32)
        mask = tf.expand_dims(mask, axis=-1)

        emb = emb * mask

        summed = tf.reduce_sum(emb, axis=1)
        denom = tf.reduce_sum(mask, axis=1)

        denom = tf.maximum(denom, 1.0)

        return summed / denom


In [54]:
# embedding dimensions

array_emb_dims = {
    "last_k_items": 32,
    "pre_purchased": 32,
    "top_10_interacted_items": 32,
    "last_C_categories": 16,
    "top_5_interacted_cats": 16,
    "favourite_3_hours": 4,
}

scalar_emb_dims = {
    "user_history_bucket": 4,
    "day": 4,
    "hour_of_day": 4,
    "favourite_day": 4,
}

item_emb_dims = {
    "itemid": 16,
    "categoryid": 16,
    "parent_category_id": 8,
    "popularity_bucket": 4,
    "generality_bucket": 4,
}


In [55]:
def weighted_multitask_bce(y_true, y_pred_logits):
    bce = tf.nn.sigmoid_cross_entropy_with_logits(
        labels=y_true,
        logits=y_pred_logits
    )

    task_weights = tf.constant([1.0, 3.0, 6.0], dtype=tf.float32)
    weighted_bce = bce * task_weights

    return tf.reduce_mean(weighted_bce)


#### Model Code

In [58]:
def build_two_tower_cg_model(lookup_layers, tower_dim=64,learning_rate=1e-3):
    inputs = {}

    # -------------------------
    # Query tower: array features
    # -------------------------
    query_parts = []

    for col in array_cols:
        inp = layers.Input(shape=(item_len_dic[col],), dtype=tf.string, name=col)
        inputs[col] = inp

        ids = lookup_layers[col](inp)

        emb = layers.Embedding(
            input_dim=lookup_layers[col].vocabulary_size(),
            output_dim=array_emb_dims[col],
            name=f"{col}_embedding"
        )(ids)

        pooled = MaskedAveragePooling(name=f"{col}_masked_avg")([emb, ids])
        query_parts.append(pooled)

    # -------------------------
    # Query tower: numeric features
    # -------------------------
    query_numeric_input = layers.Input(shape=(len(query_numeric_cols),), dtype=tf.float32, name="query_numeric")
    inputs["query_numeric"] = query_numeric_input

    query_numeric_norm = layers.BatchNormalization(name="query_numeric_norm")(query_numeric_input)

    query_parts.append(query_numeric_norm)

    # -------------------------
    # Query tower: scalar categorical features
    # -------------------------
    for col in query_scalar_cat_cols:
        inp = layers.Input(shape=(), dtype=tf.string, name=col)
        inputs[col] = inp

        ids = lookup_layers[col](inp)

        emb = layers.Embedding(
            input_dim=lookup_layers[col].vocabulary_size(),
            output_dim=scalar_emb_dims[col],
            name=f"{col}_embedding"
        )(ids)

        emb = layers.Flatten(name=f"{col}_flatten")(emb)
        query_parts.append(emb)

    query_x = layers.Concatenate(name="query_concat")(query_parts)

    query_x = layers.Dense(tower_dim*2*2, activation="relu", name="query_dense_128")(query_x)
    query_x = layers.Dense(tower_dim*2, activation="relu", name="query_dense_64")(query_x)

    q_view = layers.Dense(tower_dim, activation="linear", name="q_view")(query_x)
    q_atc = layers.Dense(tower_dim, activation="linear", name="q_atc")(query_x)
    q_order = layers.Dense(tower_dim, activation="linear", name="q_order")(query_x)

    # -------------------------
    # Item tower
    # -------------------------
    item_parts = []

    for col in item_cat_cols:
        inp = layers.Input(shape=(), dtype=tf.string, name=col)
        inputs[col] = inp

        ids = lookup_layers[col](inp)

        emb = layers.Embedding(
            input_dim=lookup_layers[col].vocabulary_size(),
            output_dim=item_emb_dims[col],
            name=f"{col}_embedding"
        )(ids)

        emb = layers.Flatten(name=f"{col}_flatten")(emb)
        item_parts.append(emb)

    item_x = layers.Concatenate(name="item_concat")(item_parts)

    item_x = layers.Dense(tower_dim*2*2, activation="relu", name="item_dense_128")(item_x)
    item_x = layers.Dense(tower_dim*2, activation="relu", name="item_dense_64")(item_x)

    item_vector = layers.Dense(tower_dim, activation="linear", name="item_vector")(item_x)

    # -------------------------
    # Multi-task dot products
    # -------------------------
    view_logit = layers.Dot(axes=1, name="view_dot")([q_view, item_vector])
    atc_logit = layers.Dot(axes=1, name="atc_dot")([q_atc, item_vector])
    order_logit = layers.Dot(axes=1, name="order_dot")([q_order, item_vector])

    logits = layers.Concatenate(name="task_logits")([
        view_logit,
        atc_logit,
        order_logit
    ])

    full_model = Model(
        inputs=inputs,
        outputs=logits,
        name="two_tower_multitask_cg"
    )

    query_encoder = Model(
        inputs={col: inputs[col] for col in array_cols + ["query_numeric"] + query_scalar_cat_cols},
        outputs=[q_view, q_atc, q_order],
        name="query_encoder"
    )

    item_encoder = Model(
        inputs={col: inputs[col] for col in item_cat_cols},
        outputs=item_vector,
        name="item_encoder"
    )

    full_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=weighted_multitask_bce,
        metrics=[
            tf.keras.metrics.AUC(name="roc_auc", from_logits=True, multi_label=True, num_labels=3),
            tf.keras.metrics.AUC(curve="PR", name="pr_auc", from_logits=True, multi_label=True, num_labels=3),
        ]
    )

    return full_model, query_encoder, item_encoder


In [62]:
# model training

two_tower_model, query_encoder, item_encoder = build_two_tower_cg_model(
    lookup_layers=lookup_layers,
    tower_dim=8,
    learning_rate=1e-3
)

two_tower_model.summary()

# If you created val_ds, you can use:
history = two_tower_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Model: "two_tower_multitask_cg"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ last_k_items        │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ last_C_categories   │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pre_purchased       │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ favourite_3_hours   │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ top_10_interacted_… │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ top_5_interacted_c… │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_history_bucket │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day (InputLayer)    │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_of_day         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ favourite_day       │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ itemid (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ categoryid          │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ parent_category_id  │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ popularity_bucket   │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ generality_bucket   │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ last_k_items_lookup │ (None, 20)        │          0 │ last_k_items[0][… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ last_C_categories_… │ (None, 5)         │          0 │ last_C_categorie… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 10,652,228 (40.64 MB)

 Trainable params: 10,652,220 (40.63 MB)

 Non-trainable params: 8 (32.00 B)

Epoch 1/5
5371/5371 ━━━━━━━━━━━━━━━━━━━━ 474s 86ms/step - loss: 0.1187 - pr_auc: 0.0576 - roc_auc: 0.7554 - val_loss: 0.1061 - val_pr_auc: 0.0992 - val_roc_auc: 0.8219
Epoch 2/5
5371/5371 ━━━━━━━━━━━━━━━━━━━━ 466s 87ms/step - loss: 0.0815 - pr_auc: 0.2080 - roc_auc: 0.9110 - val_loss: 0.1163 - val_pr_auc: 0.0943 - val_roc_auc: 0.7867
Epoch 3/5
5371/5371 ━━━━━━━━━━━━━━━━━━━━ 460s 86ms/step - loss: 0.0683 - pr_auc: 0.3123 - roc_auc: 0.9502 - val_loss: 0.1434 - val_pr_auc: 0.0913 - val_roc_auc: 0.7780
Epoch 4/5
5371/5371 ━━━━━━━━━━━━━━━━━━━━ 475s 88ms/step - loss: 0.0599 - pr_auc: 0.4057 - roc_auc: 0.9638 - val_loss: 0.1737 - val_pr_auc: 0.0816 - val_roc_auc: 0.7387
Epoch 5/5
5371/5371 ━━━━━━━━━━━━━━━━━━━━ 473s 88ms/step - loss: 0.0535 - pr_auc: 0.4867 - roc_auc: 0.9723 - val_loss: 0.1833 - val_pr_auc: 0.0813 - val_roc_auc: 0.7236


#### DataFrames for Retreival

##### Query

In [63]:

def prepare_query_encoder_df(query_df):
    df = query_df.copy()

    for col in array_cols:
        if col not in df.columns:
            df[col] = [[] for _ in range(len(df))]

        df[col] = df[col].apply(
            lambda x: pad_or_truncate(x, item_len_dic[col], PAD_TOKEN)
        )

    for col in query_numeric_cols:
        if col not in df.columns:
            df[col] = 0.0

        df[col] = df[col].fillna(0).astype("float32")

    for col in query_scalar_cat_cols:
        df[col] = safe_id_string_col(df[col])

    return df


In [64]:
def make_query_encoder_dataset(df, batch_size=2048):
    df = prepare_query_encoder_df(df)

    feature_dict = {}

    for col in array_cols:
        feature_dict[col] = np.stack(df[col].values).astype(str)

    feature_dict["query_numeric"] = df[query_numeric_cols].astype("float32").values

    for col in query_scalar_cat_cols:
        feature_dict[col] = df[col].astype(str).values

    ds = tf.data.Dataset.from_tensor_slices(feature_dict)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


##### Item

In [65]:
def prepare_item_encoder_df(active_item_df):
    df = active_item_df.copy()

    for col in item_cat_cols:
        df[col] = safe_id_string_col(df[col])

    return df

def make_item_encoder_dataset(df, batch_size=4096):
    df = prepare_item_encoder_df(df)

    feature_dict = {}

    for col in item_cat_cols:
        feature_dict[col] = df[col].astype(str).values

    ds = tf.data.Dataset.from_tensor_slices(feature_dict)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


#### Query and Item Vector Computes

In [66]:
def compute_item_vectors_for_block(block_id, item_encoder, active_items_by_block):
    item_df = active_items_by_block[block_id].copy()
    item_df = prepare_item_encoder_df(item_df)

    item_ds = make_item_encoder_dataset(item_df, batch_size=4096)

    item_vectors = item_encoder.predict(item_ds, verbose=1).astype("float32")
    item_ids = item_df["itemid"].astype(str).values

    return item_ids, item_vectors


In [67]:
def compute_query_final_vectors(query_df, query_encoder, batch_size=2048,
    view_w=1.0, atc_w=3.0, order_w=6.0):
    
    qdf = prepare_query_encoder_df(query_df)
    qds = make_query_encoder_dataset(qdf, batch_size=batch_size)

    q_view, q_atc, q_order = query_encoder.predict(qds, verbose=1)

    q_final = (
        view_w * q_view
        + atc_w * q_atc
        + order_w * q_order
    ).astype("float32")

    return q_final


In [68]:
# retreiving top k item ids from query vectors

def retrieve_topk_dot_product(
    query_vectors,
    item_vectors,
    item_ids, top_k=500, query_chunk_size=512
):
    all_recs = []
    item_matrix = item_vectors.T

    for start in range(0, len(query_vectors), query_chunk_size):
        end = min(start + query_chunk_size, len(query_vectors))

        scores = query_vectors[start:end] @ item_matrix

        kth = min(top_k, scores.shape[1]) - 1

        top_idx_unsorted = np.argpartition(
            -scores,
            kth=kth,
            axis=1
        )[:, :top_k]

        top_scores_unsorted = np.take_along_axis(
            scores,
            top_idx_unsorted,
            axis=1
        )

        sort_order = np.argsort(-top_scores_unsorted, axis=1)

        top_idx_sorted = np.take_along_axis(
            top_idx_unsorted,
            sort_order,
            axis=1
        )

        for row_idx in range(top_idx_sorted.shape[0]):
            rec_items = item_ids[top_idx_sorted[row_idx]].tolist()
            all_recs.append(rec_items)

        print(f"Retrieved queries {start:,} to {end:,}")

    return all_recs


### Validations and Test

In [69]:
# Validation
val_item_ids, val_item_vectors = compute_item_vectors_for_block(
    block_id=6,
    item_encoder=item_encoder,
    active_items_by_block=active_items_by_block
)

val_query_vectors = compute_query_final_vectors(
    val_query_df,
    query_encoder=query_encoder
)

val_two_tower_recs = retrieve_topk_dot_product(
    query_vectors=val_query_vectors,
    item_vectors=val_item_vectors,
    item_ids=val_item_ids,
    top_k=500,
    query_chunk_size=512
)

val_query_eval = val_query_df.copy()
val_query_eval["two_tower_CG"] = val_two_tower_recs

# Test:
test_item_ids, test_item_vectors = compute_item_vectors_for_block(
    block_id=7,
    item_encoder=item_encoder,
    active_items_by_block=active_items_by_block
)

test_query_vectors = compute_query_final_vectors(
    test_query_df,
    query_encoder=query_encoder
)

test_two_tower_recs = retrieve_topk_dot_product(
    query_vectors=test_query_vectors,
    item_vectors=test_item_vectors,
    item_ids=test_item_ids,
    top_k=500,
    query_chunk_size=512
)

test_query_eval = test_query_df.copy()
test_query_eval["two_tower_CG"] = test_two_tower_recs


13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step
Retrieved queries 0 to 512
Retrieved queries 512 to 1,024
Retrieved queries 1,024 to 1,536
Retrieved queries 1,536 to 2,048
Retrieved queries 2,048 to 2,560
Retrieved queries 2,560 to 3,072
Retrieved queries 3,072 to 3,584
Retrieved queries 3,584 to 4,096
Retrieved queries 4,096 to 4,608
Retrieved queries 4,608 to 5,120
Retrieved queries 5,120 to 5,632
Retrieved queries 5,632 to 6,144
Retrieved queries 6,144 to 6,656
Retrieved queries 6,656 to 7,168
Retrieved queries 7,168 to 7,680
Retrieved queries 7,680 to 8,192
Retrieved queries 8,192 to 8,704
Retrieved queries 8,704 to 9,216
Retrieved queries 9,216 to 9,728
Retrieved queries 9,728 to 10,240
Retrieved queries 10,240 to 10,752
Retrieved queries 10,752 to 11,264
Retrieved queries 11,264 to 11,776
Retrieved queries 11,776 to 12,288
Retrieved queries 12,288 to 12,800
Retrieved queries 12,800 to 13,312
Retrieved queries 13,312 to 13,824
Retrieved queries 13,8

### Evaluation

In [70]:
def evaluate_cg_list(df, pred_col, true_col, ks=[100, 200, 500]):
    rows = []

    for k in ks:
        hits, recalls, mrrs, precisions = [], [], [], []

        for _, row in df.iterrows():
            preds = dedupe_keep_order(row[pred_col])[:k]
            trues = dedupe_keep_order(row[true_col])

            if len(trues) == 0:
                continue

            pred_set = set(preds)
            true_set = set(trues)

            hit_items = pred_set & true_set

            hit = 1.0 if len(hit_items) > 0 else 0.0
            recall = len(hit_items) / len(true_set)
            precision = len(hit_items) / k

            rr = 0.0
            for idx, item in enumerate(preds, start=1):
                if item in true_set:
                    rr = 1.0 / idx
                    break

            hits.append(hit)
            recalls.append(recall)
            precisions.append(precision)
            mrrs.append(rr)

        rows.append({
            "pred_col": pred_col,
            "true_col": true_col,
            "K": k,
            "n_queries": len(hits),
            "HitRate": np.mean(hits) if hits else 0,
            "Recall": np.mean(recalls) if recalls else 0,
            "Precision": np.mean(precisions) if precisions else 0,
            "MRR": np.mean(mrrs) if mrrs else 0
        })

    return pd.DataFrame(rows)

# Validation evaluation:
val_neural_interaction_metrics = evaluate_cg_list(
    val_query_eval,
    pred_col="two_tower_CG",
    true_col="future_positives",
    ks=[100, 200, 500]
)

val_neural_engagement_metrics = evaluate_cg_list(
    val_query_eval,
    pred_col="two_tower_CG",
    true_col="future_engagement_items",
    ks=[100, 200, 500]
)

val_neural_order_metrics = evaluate_cg_list(
    val_query_eval,
    pred_col="two_tower_CG",
    true_col="future_order_items",
    ks=[100, 200, 500]
)

pd.concat([
    val_neural_interaction_metrics,
    val_neural_engagement_metrics,
    val_neural_order_metrics
], ignore_index=True)


,pred_col,true_col,K,n_queries,HitRate,Recall,Precision,MRR
0,two_tower_CG,future_positives,100,30000,0.024900,0.019297,0.000271,0.002010
1,two_tower_CG,future_positives,200,30000,0.037500,0.028796,0.000205,0.002100
2,two_tower_CG,future_positives,500,30000,0.064733,0.050292,0.000144,0.002185
3,two_tower_CG,future_engagement_items,100,1233,0.038118,0.028335,0.000414,0.003057
4,two_tower_CG,future_engagement_items,200,1233,0.055150,0.042542,0.000308,0.003174
5,two_tower_CG,future_engagement_items,500,1233,0.094079,0.075927,0.000216,0.003296
6,two_tower_CG,future_order_items,100,537,0.035382,0.026510,0.000391,0.002890
7,two_tower_CG,future_order_items,200,537,0.050279,0.036830,0.000279,0.002989
8,two_tower_CG,future_order_items,500,537,0.087523,0.069527,0.000205,0.003106


In [71]:
# Test evaluation:
test_neural_interaction_metrics = evaluate_cg_list(
    test_query_eval,
    pred_col="two_tower_CG",
    true_col="future_positives",
    ks=[100, 200, 500]
)

test_neural_engagement_metrics = evaluate_cg_list(
    test_query_eval,
    pred_col="two_tower_CG",
    true_col="future_engagement_items",
    ks=[100, 200, 500]
)

test_neural_order_metrics = evaluate_cg_list(
    test_query_eval,
    pred_col="two_tower_CG",
    true_col="future_order_items",
    ks=[100, 200, 500]
)

pd.concat([
    test_neural_interaction_metrics,
    test_neural_engagement_metrics,
    test_neural_order_metrics
], ignore_index=True)

,pred_col,true_col,K,n_queries,HitRate,Recall,Precision,MRR
0,two_tower_CG,future_positives,100,30000,0.016133,0.011967,0.000174,0.000791
1,two_tower_CG,future_positives,200,30000,0.025200,0.018588,0.000138,0.000855
2,two_tower_CG,future_positives,500,30000,0.047500,0.036390,0.000107,0.000923
3,two_tower_CG,future_engagement_items,100,1149,0.020017,0.014583,0.000218,0.000613
4,two_tower_CG,future_engagement_items,200,1149,0.034813,0.027449,0.000191,0.000718
5,two_tower_CG,future_engagement_items,500,1149,0.058312,0.047416,0.000131,0.000788
6,two_tower_CG,future_order_items,100,508,0.025591,0.018318,0.000295,0.000811
7,two_tower_CG,future_order_items,200,508,0.043307,0.034175,0.000256,0.000934
8,two_tower_CG,future_order_items,500,508,0.064961,0.052521,0.000157,0.001003
